# Refined qwen server to match interface

This notebook implements and evaluates a two-stage LLM-based binary classifier for ontology entity matching.

Source files for the experimental pipeline are available at https://github.com/city-artificial-intelligence/rai-ukraine-kga-llm.

The notebook proceeds in three main phases. First, it downloads and extracts the experimental source files, installs dependencies, and loads the `Qwen3-30B-A3B-Instruct` model in GGUF format via `llama-cpp-python`, configured for CPU inference with a 4096-token context window and full logit access. Second, it defines the `QwenServerInProcess` class, which implements a two-stage inference strategy: a hidden chain-of-thought reasoning pass followed by a constrained single-token decision pass that compares log-probabilities of the `True` and `False` tokens under a strong logit bias, yielding deterministic binary predictions. Third, it runs the full evaluation loop over the `anatomy/human-mouse` dataset subset, applying multiple prompt strategies.

\
**Note:** Use this notebook locally or in Google Colab/Kaggle

### Download source files to run experiments

In [1]:
!gdown 1s3JCs1BLKJOqWRSf1fiHfsNfi9ks-gr2 -q

In [5]:
%%capture
!unzip files_for_colab_kgllm.zip -d .

In [9]:
!mv files_for_colab_kgllm/* .
!rm -rf files_for_colab_kgllm.zip files_for_colab_kgllm

In [11]:
!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 67.4 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 45.3 MB/s eta 0:00:00


In [12]:
!pip install -U llama-cpp-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 16.0 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00


### Download model locally to run on cpu

In [ ]:
from llama_cpp import Llama
from src.constants import (
    BinaryOutputFormat,
    LLMCallOutput,
    TokensUsage,
)

llm = Llama.from_pretrained(
    repo_id="unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF",
    filename="Qwen3-30B-A3B-Instruct-2507-Q6_K.gguf",
    n_ctx=4096,
    n_threads=8,
    logits_all=True,
    verbose=False,
)

In [14]:
import math
from src.constants import BinaryOutputFormat, LLMCallOutput, TokensUsage


class QwenServerInProcess:
    def __init__(self, llm):
        self.llm = llm

        # Leading-space tokens (critical)
        self.true_token  = self.llm.tokenize(b" True",  add_bos=False)[0]
        self.false_token = self.llm.tokenize(b" False", add_bos=False)[0]

        self.logit_bias = {
            self.true_token:  50.0,
            self.false_token: 50.0,
        }

    def ask_sync_question(self, prompt: str, model: str = None) -> LLMCallOutput:
        # ---- STEP 1: reasoning (hidden) ----
        reasoning_prompt = (
            "<|system|>\n"
            "You are an expert ontology matching assistant.\n"
            "Think step by step, but do not reveal your reasoning.\n"
            "<|user|>\n"
            f"{prompt}\n"
            "<|assistant|>\n"
        )

        reasoning = self.llm.create_completion(
            prompt=reasoning_prompt,
            max_tokens=128,     # allow reasoning
            temperature=0.0,
            stop=["True", "False"],  # prevent premature answer
        )

        reasoning_text = reasoning["choices"][0]["text"]


        # ---- STEP 2: constrained decision ----
        decision_prompt = (
            reasoning_prompt
            + reasoning_text
            + "\nAnswer with exactly one word: True or False.\n"
        )

        result = self.llm.create_completion(
            prompt=decision_prompt,
            max_tokens=1,
            temperature=0.0,
            logit_bias=self.logit_bias,
            logprobs=5,
            stop=["\n"],
        )

        top_logprobs = result["choices"][0]["logprobs"]["top_logprobs"][0]

        true_score  = top_logprobs.get(" True",  -math.inf)
        false_score = top_logprobs.get(" False", -math.inf)

        if true_score == -math.inf and false_score == -math.inf:
            raise RuntimeError(f"No True/False in top_logprobs: {top_logprobs}")

        answer = "True" if true_score > false_score else "False"

        print("# answer: ", answer)

        return LLMCallOutput(
            message=answer,
            usage=TokensUsage(input_tokens=None, output_tokens=None),
            logprobs=[],
            parsed=BinaryOutputFormat(answer=answer == "True"),
        )


qwen = QwenServerInProcess(llm)


### Simple examples

In [12]:
a = """
The first one is ""stomach muscularis mucosa"", which falls under the category ""Thing"".
The second one is ""Gastric_Muscularis_Mucosa"", also known as ""Gastric Muscularis Mucosa"", which falls under the category ""Gastric_Mucosa"".

Do they mean the same thing? Respond with ""True"" or ""False""."
"""

qwen.ask_sync_question(a)

# answer:  True


LLMCallOutput(message='True', usage=TokensUsage(input_tokens=None, output_tokens=None), logprobs=[], parsed=BinaryOutputFormat(answer=True))

In [21]:
qwen.ask_sync_question("Is 2 + 2 equal to 4?") # -> True
qwen.ask_sync_question("Do you love humans?") # -> False

# True
# False


LLMCallOutput(message='False', usage=TokensUsage(input_tokens=None, output_tokens=None), logprobs=[], parsed=BinaryOutputFormat(answer=False))

### Run experiments

In [ ]:
# ruff: noqa: T201, T203, ERA001, BLE001
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

import src.prompts.system as system_prompts
from config.config import RUN_DIR
from config.runs_vars import SUBSET_TO_DATASET_MAP
from src.constants import PAIRS_SEPARATOR
from src.evaluate import (
    get_predictions_with_gt,
    plot_usage_histograms,
    read_run_metrics_df,
    save_analysis_results,
    store_run_metrics_df,
)
from src.formatting import (
    format_oracle_pairs_filepath,
    format_oracle_pairs_precomputed_dir,
    format_predictions_run_path,
    format_run_path,
    format_storing_pathes_from_run_path,
    format_subsets_ontologies_paths,
)
from src.LLM_servers.openai import OpenAIServer
from src.LLM_servers.qwen import QwenServer

from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.processing import parallel_samples_process, save_oracle_pairs_with_prompts, try_load_precomputed_oracle_pairs
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)
from src.utils import read_oracle_pairs, save_run_results

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

In [ ]:
MAX_WORKERS = 1
MODELS = [
    # "gemini-2.0-flash-lite",
    # "gpt-4.1-nano",
    # "qwen3-1.7B",
    # "qwen3-8B",
    # "Qwen/Qwen1.5-MoE-A2.7B",
    "Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF",
]  # "gpt-4o-mini", "gemini-2.5-flash-preview-04-17", "gemini-2.0-flash-lite", "gemini-1.5-flash", "gemini-2.0-flash", "qwen3-1.7B"

DATASETS_MAP = {
    # "largebio": ["fma-nci"],  # , "snomed-nci", "fma-snomed"],
    # "bioml-2024": ["omim-ordo",] #  "snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid"]
    "anatomy": ["human-mouse"],
}
EXPERIMENT_TYPE = "prompts"

from src.constants import BinaryOutputFormat, BinaryOutputFormatWithReasoning

# EXP_NAMES_MAP = {"present": BinaryOutputFormat, "absent": BinaryOutputFormatWithReasoning}
EXP_NAMES_MAP = {"present": BinaryOutputFormat}

onto_src, onto_tgt = None, None

PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        prompt_direct_entity,
        prompt_direct_entity_ontological,
        prompt_direct_entity_with_synonyms,
        prompt_sequential_hierarchy,
        prompt_sequential_hierarchy_ontological,
        prompt_sequential_hierarchy_with_synonyms,
    ]
}

RUNS = []

In [ ]:
for DATASET in DATASETS_MAP:
    for SUBSET in DATASETS_MAP[DATASET]:
        # Load the ontologies here, if not using precomputed prompts
        # src_onto_path, tgt_onto_path = format_subsets_ontologies_paths(DATASET, SUBSET)
        # onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
        # onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)
        for exp_name in EXP_NAMES_MAP:
            if exp_name == "present" and SUBSET == "omin-ordo":
                continue
            for MODEL in MODELS:
                llm_oracle = qwen

                # llm_oracle.add_system_context(system_prompts.INTUITIVE_NATURAL_LANGUAGE_JUDGEMENT_MESSAGE)

                run_path = (
                    format_predictions_run_path(DATASET, SUBSET, MODEL, EXPERIMENT_TYPE, exp_spec=exp_name)
                    if exp_name
                    else format_run_path()
                )
                print(f"{run_path=} | {DATASET=} | {SUBSET=} | {MODEL=} | {exp_name=}")

                for prompt_name, prompt_function in PROMPT_FUNCTIONS_MAP.items():
                    prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
                        run_path, SUBSET, MODEL, prompt_name, suffix=""
                    )
                    oracle_candidate_pairs = try_load_precomputed_oracle_pairs(
                        DATASET, SUBSET, prompt_function, suffix=""
                    )
                    results, tokens_usage, confidences = parallel_samples_process(
                        oracle_candidate_pairs, llm_oracle, onto_src, onto_tgt, MODEL, MAX_WORKERS, prompt_function
                    )

                    save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])
                    plot_usage_histograms(
                        tokens_usage, confidences, do_plot=False, do_print=False, suptitle=prompt_name
                    )
                    try:
                        predictions = get_predictions_with_gt(run_path, DATASET, SUBSET, MODEL, prompt_name, suffix="")
                        save_analysis_results(
                            predictions,
                            print_results=False,
                            plot_confusion_matrix=False,
                            subtitle=f"{SUBSET}: {MODEL} {prompt_name} | ",
                            cm_save_path=diagram_path,
                            stats_path=stats_path,
                        )
                    except Exception as e:
                        print(f"Error: {e}")

                store_run_metrics_df(PROMPT_FUNCTIONS_MAP, run_path, DATASET, SUBSET, MODEL)


zip the Qwen model output directory into a single archive for download or sharing

In [18]:
!zip -r /kaggle/working/Qwen3-30B-A3B-Instruct-2507-GGUF.zip /kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF

updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/ (stored 0%)
updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/prompts/ (stored 0%)
updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/prompts/present/ (stored 0%)
updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/prompts/present/results.csv (deflated 19%)
updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/prompts/present/omim-ordo_Qwen/ (stored 0%)
updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/prompts/present/omim-ordo_Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF_sequential_hierarchy_with_synonyms_results.csv (deflated 90%)
updating: kaggle/working/outputs/bioml-2024/omim-ordo/Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF/prompts/present/omim-ordo_Qwen/Qwen3-30B-A3B-Instruct-2507-GGUF_sequential_hiera